In [1]:
import collections
import json
import os
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch


warnings.filterwarnings("ignore")  # avoid torchmetrics warning

import jax
import jax.numpy as jnp
from disk.data.mapper import map_frame
from disk.data.record import RecordLoader
from disk.data.stats import TextStats
from disk.dataset.wikidata_names import WikidataNames
from disk.model.disk import DiSK

from diffusion_linking.smc_clustering import (
    BigramCluster,
    BigramMixture,
    DirichletProcess,
    SMCClusterer,
    resample_optimal,
    resample_stratified,
)
from diffusion_linking.utils import DFWrapper

In [2]:
def estimate_probs(
    df: pd.DataFrame,
    model: DiSK,
    num_samples: int,
    device: str = "cpu",
    batch_size: int = 200,
    num_workers: int = 0,
    verbose: bool = False,
) -> np.ndarray:
    """
    Modification of disk.linking.score.estimate_linking_scores to return the log probs for each cluster
    """

    df.reset_index(inplace=True)

    # map to RecordData
    record_data = map_frame(df, model.schema, model.stats)

    # Build data loader if necessary
    if record_data.num_records <= batch_size:
        loader = [record_data]
    else:
        loader = RecordLoader(
            data=record_data,
            input_nodes=np.arange(record_data.num_records),
            batch_size=batch_size,
            num_workers=num_workers,
            shuffle=False,
        )

    model.to(device)
    disk_model = model.model

    # get all log probs in one array
    all_log_probs = []
    with torch.no_grad():
        for data in loader:
            data.to(device)
            log_probs = disk_model.monte_carlo_log_probs(data, num_samples=num_samples)
            log_probs = log_probs.detach().cpu().numpy()
            all_log_probs.append(log_probs)
    log_probs = np.concatenate(all_log_probs, axis=0)

    return log_probs


def score_fn(rng, cluster_data, num_samples=10, batch_size=1):
    df = pd.DataFrame.from_records([{"names": names} for names in cluster_data])
    log_probs = estimate_probs(df, model, num_samples, device, batch_size=batch_size)
    return log_probs

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
schema = WikidataNames.schema
stats = {"names": TextStats("names", "t5-small", 32128)}

checkpoint = "checkpoints/DiSK_Autoregressive/checkpoints/best.ckpt"
model = DiSK.load_from_checkpoint(checkpoint, schema=schema, stats=stats)
model.to(device)
model.model.use_diffusion_weights = True

Load first part of the Wikinames dataset.

In [4]:
data = []
with open("data/candidates_val.jsonl") as file:
    for line in file:
        entry = json.loads(line)
        names = entry["entity1"]["properties"]["names"] + entry["entity2"]["properties"]["names"]
        for name in names:
            if name not in data:
                data.append(name)

dataset = DFWrapper(pd.DataFrame.from_records([{"name": name} for name in [""] + data[:500]]))

Load prior model pretrained on Wikipedia titles. This is the prior over bigram counts for the surrogate model.

In [5]:
with open(f"data/wikipedia_names_ngram_counts.pickle", "rb") as handle:
    count_dict = pickle.load(handle)
prior_counts = collections.defaultdict(lambda: count_dict["<UNK>"], count_dict)
surrogate = BigramMixture(0.01, prior_counts)

Run clusterer.

In [6]:
rng = jax.random.PRNGKey(1)
prior = DirichletProcess(10)
max_particles = 20
max_evals = 20
resample_fn = resample_stratified
resample_inner = resample_stratified
clusterer = SMCClusterer(
    data=dataset,
    split_interval=1,
    threshold=10,
    score_fn=score_fn,
    max_particles=max_particles,
    max_evals=max_evals,
    prior=prior,
    surrogate=surrogate,
    resample_fn=resample_fn,
    resample_inner=resample_inner,
    ClusterClass=BigramCluster,
)
clusterer.cluster(rng, verbose=False)

100%|████████████████████████| 499/499 [17:17<00:00,  2.08s/it, Subproblems=298]

Summary of larger subproblems:

In [7]:
clusterer.summary(print_cluster_data=True, min_problem_size=4, max_print=3)

298 subproblems of sizes [6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Subproblem 3: 10 particles, 5 points
	Particle 0, weight 1(0), 4 clusters, [2, 1

"298 subproblems of sizes [6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]\nSubproblem 3: 10 particles, 5 points\n\tParticle 0, weight 1(0), 4 clusters, [

Full clustering:

In [8]:
clusterer.summary(print_cluster_data=True)

298 subproblems of sizes [6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Subproblem 0: 2 particles, 2 points
	Particle 0, weight 1(0), 1 clusters, [2]
		

'298 subproblems of sizes [6, 6, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]\nSubproblem 0: 2 particles, 2 points\n\tParticle 0, weight 1(0), 1 clusters, [2